# KNN 回归：用邻居的均值做预测
> 难度标签：初级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：02_监督学习/回归 | 源文件：KNN_回归.py | 核心功能：K 值选择、距离加权、距离度量对比

## 概述

KNN 回归是 KNN 分类的回归版本——找到 K 个最近邻，取它们目标值的平均（或加权平均）作为预测。没有训练过程，所有计算都在预测时完成。

脚本对比了不同 K 值、权重策略和距离度量对回归性能的影响。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
# --- 导入库 ---
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

## 数学原理

### 1. KNN 回归的预测公式

**代码对应**：`KNeighborsRegressor(n_neighbors=k, weights="uniform")` 和 `weights="distance"`。

对于查询样本 $\mathbf{x}$，找到 $K$ 个最近邻 $\mathcal{N}_K(\mathbf{x}) = \{i_1, i_2, \ldots, i_K\}$，预测值为：

**等权平均**（`weights="uniform"`）：

$$\hat{y}(\mathbf{x}) = \frac{1}{K}\sum_{i \in \mathcal{N}_K(\mathbf{x})} y_i$$

**距离加权平均**（`weights="distance"`）：

$$\hat{y}(\mathbf{x}) = \frac{\sum_{i \in \mathcal{N}_K(\mathbf{x})} w_i y_i}{\sum_{i \in \mathcal{N}_K(\mathbf{x})} w_i}, \quad w_i = \frac{1}{d(\mathbf{x}, \mathbf{x}_i)}$$

**代码对应**：代码中对比了 `uniform` 和 `distance` 两种权重策略。距离加权让更近的邻居有更大影响，通常效果更好。

### 2. 距离度量

**代码对应**：`metric="minkowski", p=p` 定义了 Minkowski 距离族。

闵可夫斯基距离：

$$d_p(\mathbf{x}, \mathbf{z}) = \left(\sum_{j=1}^{p} |x_j - z_j|^p\right)^{1/p}$$

- $p = 1$：曼哈顿距离（L1）
- $p = 2$：欧氏距离（L2，最常用）
- $p \to \infty$：切比雪夫距离

### 3. K 的偏差-方差权衡

**代码对应**：`for k in [1, 3, 5, ..., 50]` 展示了 K 对训练/测试 R² 的影响。

- $K = 1$：预测值等于最近邻的 $y$ 值。训练集上完美拟合（$R^2 = 1$），但泛化极差（高方差）
- $K = N$：预测值始终为全局均值 $\bar{y}$（高偏差，零方差）
- 最优 $K$ 在两者之间，通过交叉验证选择

**数学分析**：KNN 回归的期望误差可以分解为：

$$\mathbb{E}[(y - \hat{y})^2] = \underbrace{\left(\mathbb{E}[\hat{y}] - f(\mathbf{x})\right)^2}_{\text{Bias}^2} + \underbrace{\text{Var}[\hat{y}]}_{\text{Variance}} + \sigma^2$$

- $K$ 增大 $\Rightarrow$ 偏差增大（邻居平均值趋近全局均值），方差减小
- $K$ 减小 $\Rightarrow$ 偏差减小，方差增大

### 4. 维度灾难

KNN 在高维空间中失效。设数据均匀分布在 $[0,1]^p$ 的单位超立方体中，$K$ 个最近邻到查询点的平均距离为：

$$d_K \sim \left(\frac{K}{N}\right)^{1/p}$$

当 $p$ 很大时，即使 $N = 10000$，$K = 5$，最近邻的平均距离也接近 1——即最近邻和最远邻"一样远"。此时 KNN 的预测质量急剧下降。

**实际建议**：$p > 20$ 时 KNN 效果开始明显下降，考虑降维或使用其他模型。

### 5. 训练与预测复杂度

| 阶段 | 朴素实现 | KD-Tree（$p < 20$） | Ball Tree |
|------|---------|---------------------|-----------|
| 训练 | $O(1)$（存储即可） | $O(np\log n)$ | $O(np\log n)$ |
| 预测（每个样本） | $O(np)$ | $O(p\log n)$ | $O(p\log n)$ |

KNN 是**懒惰学习**（lazy learning）的典型代表：训练阶段几乎不做计算，所有计算推迟到预测阶段。这意味着预测速度慢，不适合实时应用。

### 1. 生成数据

生成合成数据集，用于演示算法的核心行为。

In [ ]:
X, y = make_regression(n_samples=300, n_features=5, n_informative=3, noise=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# KNN 基于距离，必须缩放
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### 2. K 值选择

运行 2. K 值选择 的代码，观察算法在该环节的行为。

In [ ]:
print("=== K 值选择 ===")
for k in [1, 3, 5, 7, 10, 15, 20, 30, 50]:
    knn = KNeighborsRegressor(n_neighbors=k, weights="uniform")
    knn.fit(X_train, y_train)
    train_r2 = knn.score(X_train, y_train)
# --- 继续 ---
    test_r2 = knn.score(X_test, y_test)
    print(f"  K={k:>2}: 训练R²={train_r2:.4f}, 测试R²={test_r2:.4f}")

### 3. 权重策略

运行 3. 权重策略 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 权重策略对比 ===")
for weights in ["uniform", "distance"]:
    knn_w = KNeighborsRegressor(n_neighbors=5, weights=weights)
    knn_w.fit(X_train, y_train)
    test_r2 = knn_w.score(X_test, y_test)
# --- 模型预测 ---
    y_pred = knn_w.predict(X_test)
    print(f"  {weights:>10}: R²={test_r2:.4f}, MSE={mean_squared_error(y_test, y_pred):.4f}")

### 4. 距离度量

运行 4. 距离度量 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 距离度量对比 ===")
for p, name in [(1, "曼哈顿"), (2, "欧氏"), (3, "闵可夫斯基-3")]:
    knn_d = KNeighborsRegressor(n_neighbors=5, metric="minkowski", p=p)
    knn_d.fit(X_train, y_train)
    test_r2 = knn_d.score(X_test, y_test)
# --- 输出结果 ---
    print(f"  {name:>12} (p={p}): R²={test_r2:.4f}")

### 5. 最终模型与评估

在测试集上评估模型性能，关注关键指标。

In [ ]:
print("\n=== 最终模型 (K=5, distance) ===")
knn_final = KNeighborsRegressor(n_neighbors=5, weights="distance", p=2)
knn_final.fit(X_train, y_train)
y_pred = knn_final.predict(X_test)

print(f"测试 R²: {r2_score(y_test, y_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")

print("\n=== 预测值 vs 真实值（前 10 个）===")
for i in range(10):
    print(f"  真实={y_test[i]:>8.2f}, 预测={y_pred[i]:>8.2f}")

print("\n=== KNN 回归要点 ===")
print("- 预测值 = K 个邻居目标值的平均（uniform）或加权平均（distance）")
print("- K=1 时完全拟合训练数据（过拟合），K 大时趋于全局均值（欠拟合）")
print("- 特征缩放必须！距离计算受特征尺度影响")
print("- 不学习显式模型，预测时计算所有训练样本距离，速度慢")
# --- 输出结果 ---
print("- 适合低维、小数据集；高维大数据效果差（维度灾难）")

## 关键代码解释

### 距离加权

```python
KNeighborsRegressor(n_neighbors=5, weights="distance")
```

uniform：简单平均。distance：权重 = 1/distance，近邻影响更大。distance 加权通常在 K 较大时表现更好。

## 使用示例

```python
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor(n_neighbors=5, weights="distance"))])
pipe.fit(X_train, y_train)
```

## 注意事项

1. **必须特征缩放**
2. **预测速度慢**：O(n·d) 每个预测样本
3. **高维失效**：维度灾难使距离变得无意义
4. **不学习显式模型**：无法获得特征重要性或系数解读

## 总结与延伸

以上代码展示了 **KNN 回归** 的完整流程。

**解读要点**：
- 关注 **R² 分数** 和 **MSE/MAE** 等回归指标
- R² 越接近 1 说明拟合越好
- 观察预测值 vs 真实值的散点图是否沿对角线分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **RadiusNeighborsRegressor**：用固定半径而非固定 K 选择邻居
- **局部加权回归（LWR）**：KNN 的平滑版本，用核函数给距离赋权
- **FAISS 等 ANN 库**：加速大规模最近邻搜索
